# DATA Processing from D6

In [ ]:
import os
import h5py
import numpy as np

RAW_PATH = "/content/drive/MyDrive/D6_raw.mat"
SAVE_DIR = "/content/drive/MyDrive/D6_temporal"
os.makedirs(SAVE_DIR, exist_ok=True)

TIME_STEPS = 4
IMG_H = 32
IMG_W = 32
KEEP_SUBCARRIERS = 125
FFT_LEN = 257


def read_h5_complex(f, key):
    obj = f[key]

    if isinstance(obj, h5py.Group):
        print(key, "is Group, keys =", list(obj.keys()))
        raise ValueError("H_transfer is Group. Need inspect inner keys.")

    data = np.array(obj)

    # MATLAB v7.3 complex format
    if data.dtype.fields is not None:
        fields = data.dtype.fields.keys()
        if "real" in fields and "imag" in fields:
            data = data["real"] + 1j * data["imag"]
        elif "r" in fields and "i" in fields:
            data = data["r"] + 1j * data["i"]
        else:
            raise ValueError(f"Unknown complex fields: {fields}")

    return data


def inspect_raw(path):
    with h5py.File(path, "r") as f:
        print("Keys:", list(f.keys()))
        for k in f.keys():
            obj = f[k]
            if isinstance(obj, h5py.Dataset):
                print(k, obj.shape, obj.dtype)
            else:
                print(k, "Group:", list(obj.keys()))


def convert_H_transfer_to_array(H):
    """
    Goal:
    Convert raw H_transfer into [sample/user, time, rx/ant, subcarrier]
    then into [N, T, 32, 125].
    """

    print("Original H_transfer shape:", H.shape)

    H = np.squeeze(H)
    print("After squeeze:", H.shape)

    # h5py usually reverses MATLAB dimensions, so try transpose first
    # You MUST check print output after this.
    H = np.transpose(H)
    print("After transpose:", H.shape)

    return H


def make_csinet_temporal_data(H):
    print("Input H shape for conversion:", H.shape)

    if H.ndim == 4:
        shape = H.shape
        print("4D shape:", shape)

        # [time, freq, users, antennas] = [50, 257, 9, 128]
        if shape[1] == 257 and shape[3] >= 32:
            Hf = np.transpose(H, (2, 0, 3, 1))
            Hf = Hf[:, :, :32, :125]

        else:
            raise ValueError(f"Cannot auto-detect 4D H shape: {shape}")

    else:
        raise ValueError(f"Unsupported H shape: {H.shape}")

    print("Frequency-domain Hf shape [users, time, 32, 125]:", Hf.shape)

    users, total_time, ant, subc = Hf.shape

    pad_width = 257 - subc
    Hf_pad = np.concatenate(
        [Hf, np.zeros((users, total_time, ant, pad_width), dtype=Hf.dtype)],
        axis=-1
    )

    H_delay = np.fft.ifft(Hf_pad, axis=-1)
    H_delay = H_delay[:, :, :, :32]

    max_abs = np.max(np.abs(H_delay)) + 1e-12
    H_delay = H_delay / (2 * max_abs) + 0.5

    real = np.real(H_delay)
    imag = np.imag(H_delay)

    data = np.stack([real, imag], axis=2)  # [users, time, 2, 32, 32]
    data = data.astype(np.float32)

    seqs, targets = [], []

    for u in range(users):
        for t in range(total_time - TIME_STEPS + 1):
            seqs.append(data[u, t:t + TIME_STEPS])
            targets.append(data[u, t + TIME_STEPS - 1])

    seqs = np.array(seqs, dtype=np.float32)
    targets = np.array(targets, dtype=np.float32)

    print("Final sequence shape:", seqs.shape)
    print("Final target shape:", targets.shape)

    return seqs, targets


# =========================================================
# Run
# =========================================================
inspect_raw(RAW_PATH)

with h5py.File(RAW_PATH, "r") as f:
    H = read_h5_complex(f, "H_transfer")

H = convert_H_transfer_to_array(H)

seqs, targets = make_csinet_temporal_data(H)

np.save(os.path.join(SAVE_DIR, "D6_seq.npy"), seqs)
np.save(os.path.join(SAVE_DIR, "D6_target.npy"), targets)

print("Saved:")
print(os.path.join(SAVE_DIR, "D6_seq.npy"))
print(os.path.join(SAVE_DIR, "D6_target.npy"))

In [2]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def print_complexity(model, model_name):
    ue_params = count_params(model.encoder)
    total_params = count_params(model)

    print(f"\n===== {model_name} Complexity =====")
    print(f"UE-side Params   : {ue_params:,}")
    print(f"Full Model Params: {total_params:,}")
    print(f"UE / Total       : {ue_params / total_params * 100:.2f}%")

# CsiNet-LSTM

In [12]:
# =========================================================
# CsiNet-LSTM Training for Temporal COST2100 NPY Dataset
# =========================================================

import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split


# =========================================================
# 1. Config
# =========================================================
SEQ_PATH = "/content/drive/MyDrive/D6_temporal/D6_seq.npy"
TARGET_PATH = "/content/drive/MyDrive/D6_temporal/D6_target.npy"

SAVE_DIR = "./csinet_lstm_temporal_ckpt"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-5

COMPRESSION_RATIO = 16
LSTM_HIDDEN = 256
LSTM_LAYERS = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# 2. Seed
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# =========================================================
# 3. Dataset
# =========================================================
class CSITemporalDataset(Dataset):
    def __init__(self, seq_path, target_path):
        self.seq = np.load(seq_path).astype(np.float32)
        self.target = np.load(target_path).astype(np.float32)

        assert self.seq.ndim == 5, self.seq.shape
        assert self.target.ndim == 4, self.target.shape
        assert len(self.seq) == len(self.target)

        print("Sequence shape:", self.seq.shape)
        print("Target shape:", self.target.shape)

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return torch.from_numpy(self.seq[idx]), torch.from_numpy(self.target[idx])


# =========================================================
# 4. Model
# =========================================================
class CsiEncoder(nn.Module):
    def __init__(self, in_channels, height, width, compression_ratio=16):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
        )

        feature_dim = 32 * height * width
        self.code_dim = max(16, feature_dim // compression_ratio)

        self.fc = nn.Linear(feature_dim, self.code_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(1)
        return self.fc(x)


class CsiDecoder(nn.Module):
    def __init__(self, out_channels, height, width, code_dim):
        super().__init__()

        self.height = height
        self.width = width

        self.fc = nn.Linear(code_dim, 32 * height * width)

        self.decoder = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(16, out_channels, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), 32, self.height, self.width)
        return self.decoder(x)


class CsiNetLSTM(nn.Module):
    def __init__(
        self,
        in_channels=2,
        height=32,
        width=32,
        compression_ratio=16,
        lstm_hidden=256,
        lstm_layers=1
    ):
        super().__init__()

        self.encoder = CsiEncoder(
            in_channels,
            height,
            width,
            compression_ratio
        )

        code_dim = self.encoder.code_dim

        self.lstm = nn.LSTM(
            input_size=code_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True
        )

        self.proj = nn.Linear(lstm_hidden, code_dim)

        self.decoder = CsiDecoder(
            out_channels=in_channels,
            height=height,
            width=width,
            code_dim=code_dim
        )

    def forward(self, x):
        # x: [B, T, 2, 32, 32]
        B, T, C, H, W = x.shape

        z_list = []
        for t in range(T):
            z = self.encoder(x[:, t])
            z_list.append(z)

        z_seq = torch.stack(z_list, dim=1)  # [B, T, code_dim]

        out, _ = self.lstm(z_seq)
        z_final = self.proj(out[:, -1])

        recon = self.decoder(z_final)
        return recon


# =========================================================
# 5. Metrics
# =========================================================
def nmse_linear(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    mse = torch.sum(torch.abs(pred_c - target_c) ** 2, dim=1)
    power = torch.sum(torch.abs(target_c) ** 2, dim=1) + 1e-8

    return torch.mean(mse / power)


def corr_complex(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    numerator = torch.abs(torch.sum(torch.conj(target_c) * pred_c, dim=1))
    denominator = torch.sqrt(
        torch.sum(torch.abs(target_c) ** 2, dim=1) *
        torch.sum(torch.abs(pred_c) ** 2, dim=1)
    ) + 1e-8

    return torch.mean(numerator / denominator)


# =========================================================
# 6. Train / Eval
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Train", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad()

        pred = model(seq)
        loss = criterion(pred, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Eval", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        pred = model(seq)
        loss = criterion(pred, target)

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


# =========================================================
# 7. Main
# =========================================================
dataset = CSITemporalDataset(SEQ_PATH, TARGET_PATH)

sample_seq, sample_target = dataset[0]
T, C, H, W = sample_seq.shape

print("T, C, H, W =", T, C, H, W)

train_len = int(len(dataset) * 0.7)
val_len = int(len(dataset) * 0.15)
test_len = len(dataset) - train_len - val_len

train_set, val_set, test_set = random_split(
    dataset,
    [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

model = CsiNetLSTM(
    in_channels=C,
    height=H,
    width=W,
    compression_ratio=COMPRESSION_RATIO,
    lstm_hidden=LSTM_HIDDEN,
    lstm_layers=LSTM_LAYERS
).to(DEVICE)

print(model)
print("Code dim:", model.encoder.code_dim)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

best_val_nmse = float("inf")
best_path = os.path.join(SAVE_DIR, "best_csinet_lstm_temporal.pth")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_nmse, train_corr = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss, val_nmse, val_corr = evaluate(
        model, val_loader, criterion
    )

    scheduler.step()

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.6f} | "
        f"Train NMSE(dB): {10*np.log10(train_nmse + 1e-12):.4f} | "
        f"Train Corr: {train_corr:.4f} || "
        f"Val Loss: {val_loss:.6f} | "
        f"Val NMSE(dB): {10*np.log10(val_nmse + 1e-12):.4f} | "
        f"Val Corr: {val_corr:.4f}"
    )

    if val_nmse < best_val_nmse:
        best_val_nmse = val_nmse

        torch.save({
            "model_state_dict": model.state_dict(),
            "T": T,
            "C": C,
            "H": H,
            "W": W,
            "compression_ratio": COMPRESSION_RATIO,
            "lstm_hidden": LSTM_HIDDEN,
            "lstm_layers": LSTM_LAYERS,
            "best_val_nmse": best_val_nmse,
        }, best_path)

        print("Saved best model:", best_path)

print("\nLoading best model for testing...")
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_nmse, test_corr = evaluate(model, test_loader, criterion)

print("\n========== Final Test Result ==========")
print(f"Test Loss      : {test_loss:.6f}")
print(f"Test NMSE      : {test_nmse:.6f}")
print(f"Test NMSE(dB)  : {10*np.log10(test_nmse + 1e-12):.4f} dB")
print(f"Test Corr      : {test_corr:.4f}")

Sequence shape: (423, 4, 2, 32, 32)
Target shape: (423, 2, 32, 32)
T, C, H, W = 4 2 32 32
CsiNetLSTM(
  (encoder): CsiEncoder(
    (conv): Sequential(
      (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.2, inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): LeakyReLU(negative_slope=0.2, inplace=True)
    )
    (fc): Linear(in_features=32768, out_features=2048, bias=True)
  )
  (lstm): LSTM(2048, 256, batch_first=True)
  (proj): Linear(in_features=256, out_features=2048, bias=True)
  (decoder): CsiDecoder(
    (fc): Linear(in_features=2048, out_features=32768, bias=True)
    (decoder): Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32

Epoch [001/200] Train Loss: 0.084505 | Train NMSE(dB): -1.7137 | Train Corr: 0.7262 || Val Loss: 0.045505 | Val NMSE(dB): -4.4014 | Val Corr: 0.9716
Saved best model: ./csinet_lstm_temporal_ckpt/best_csinet_lstm_temporal.pth


Epoch [002/200] Train Loss: 0.038761 | Train NMSE(dB): -5.0986 | Train Corr: 0.9833 || Val Loss: 0.016163 | Val NMSE(dB): -8.8967 | Val Corr: 0.9911
Saved best model: ./csinet_lstm_temporal_ckpt/best_csinet_lstm_temporal.pth


Epoch [003/200] Train Loss: 0.019483 | Train NMSE(dB): -8.0864 | Train Corr: 0.9941 || Val Loss: 0.010710 | Val NMSE(dB): -10.6841 | Val Corr: 0.9958
Saved best model: ./csinet_lstm_temporal_ckpt/best_csinet_lstm_temporal.pth


Epoch [004/200] Train Loss: 0.010536 | Train NMSE(dB): -10.7564 | Train Corr: 0.9966 || Val Loss: 0.009132 | Val NMSE(dB): -11.3766 | Val Corr: 0.9972
Saved best model: ./csinet_lstm_temporal_ckpt/best_csinet_lstm_temporal.pth


...

Epoch [197/200] Train Loss: 0.000398 | Train NMSE(dB): -24.9840 | Train Corr: 0.9985 || Val Loss: 0.000408 | Val NMSE(dB): -24.8820 | Val Corr: 0.9985
Saved best model: ./csinet_lstm_temporal_ckpt/best_csinet_lstm_temporal.pth


Epoch [198/200] Train Loss: 0.000398 | Train NMSE(dB): -24.9895 | Train Corr: 0.9985 || Val Loss: 0.000408 | Val NMSE(dB): -24.8832 | Val Corr: 0.9985
Saved best model: ./csinet_lstm_temporal_ckpt/best_csinet_lstm_temporal.pth


Epoch [199/200] Train Loss: 0.000401 | Train NMSE(dB): -24.9603 | Train Corr: 0.9985 || Val Loss: 0.000408 | Val NMSE(dB): -24.8829 | Val Corr: 0.9985


Epoch [200/200] Train Loss: 0.000406 | Train NMSE(dB): -24.9005 | Train Corr: 0.9985 || Val Loss: 0.000408 | Val NMSE(dB): -24.8827 | Val Corr: 0.9985

Loading best model for testing...



========== Final Test Result ==========
Test Loss      : 0.000457
Test NMSE      : 0.003642
Test NMSE(dB)  : -24.3868 dB
Test Corr      : 0.9983


In [13]:
def print_csinet_lstm_ue_complexity(model):
    encoder_params = count_params(model.encoder)
    lstm_params = count_params(model.lstm)
    ue_params = encoder_params + lstm_params
    total_params = count_params(model)

    print("\n===== CsiNet-LSTM Complexity =====")
    print(f"Encoder Params       : {encoder_params:,}")
    print(f"LSTM Params          : {lstm_params:,}")
    print(f"UE-side Params       : {ue_params:,}")
    print(f"Full Model Params    : {total_params:,}")
    print(f"UE / Total           : {ue_params / total_params * 100:.2f}%")
print_csinet_lstm_ue_complexity(model)


===== CsiNet-LSTM Complexity =====
Encoder Params       : 67,120,896
LSTM Params          : 2,361,344
UE-side Params       : 69,482,240
Full Model Params    : 137,164,466
UE / Total           : 50.66%


# DA-TCsiNet (FULL)

In [14]:
# =========================================================
# DA-TCsiNet
# Multi-scale TCN + Doppler-aware Delta Gate + Last-frame Residual Fusion
# =========================================================

import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split


# =========================================================
# 1. Config
# =========================================================
SEQ_PATH = "/content/drive/MyDrive/D6_temporal/D6_seq.npy"
TARGET_PATH = "/content/drive/MyDrive/D6_temporal/D6_target.npy"

SAVE_DIR = "./datcsinet_ckpt"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-5

COMPRESSION_RATIO = 16
HIDDEN_DIM = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# 2. Seed
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# =========================================================
# 3. Dataset
# =========================================================
class CSITemporalDataset(Dataset):
    def __init__(self, seq_path, target_path):
        self.seq = np.load(seq_path).astype(np.float32)
        self.target = np.load(target_path).astype(np.float32)

        assert self.seq.ndim == 5, self.seq.shape
        assert self.target.ndim == 4, self.target.shape
        assert len(self.seq) == len(self.target)

        print("Sequence shape:", self.seq.shape)
        print("Target shape:", self.target.shape)

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return torch.from_numpy(self.seq[idx]), torch.from_numpy(self.target[idx])


# =========================================================
# 4. Encoder / Decoder
# =========================================================
class CsiEncoder(nn.Module):
    def __init__(self, in_channels, height, width, compression_ratio=16):
        super().__init__()

        self.height = height
        self.width = width

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
        )

        feature_dim = 32 * height * width
        self.code_dim = max(16, feature_dim // compression_ratio)

        self.fc = nn.Linear(feature_dim, self.code_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(1)
        return self.fc(x)


class CsiDecoder(nn.Module):
    def __init__(self, out_channels, height, width, code_dim):
        super().__init__()

        self.height = height
        self.width = width

        self.fc = nn.Linear(code_dim, 32 * height * width)

        self.decoder = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(16, out_channels, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), 32, self.height, self.width)
        return self.decoder(x)


# =========================================================
# 5. Proposed Temporal Modules
# =========================================================
class MultiScaleTCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()

        self.branch3 = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.branch5 = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=5, padding=4),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.branch_dilated = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=4, dilation=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.fuse = nn.Sequential(
            nn.Conv1d(hidden_dim * 3, hidden_dim, kernel_size=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # x: [B, T, code_dim]
        x = x.transpose(1, 2)  # [B, code_dim, T]

        y1 = self.branch3(x)
        y2 = self.branch5(x)
        y3 = self.branch_dilated(x)

        min_len = min(y1.size(-1), y2.size(-1), y3.size(-1))
        y1 = y1[:, :, -min_len:]
        y2 = y2[:, :, -min_len:]
        y3 = y3[:, :, -min_len:]

        y = torch.cat([y1, y2, y3], dim=1)
        y = self.fuse(y)

        return y[:, :, -1]  # [B, hidden_dim]


class DopplerAwareGate(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )

    def forward(self, temporal_feat, delta_feat):
        g = self.gate(torch.cat([temporal_feat, delta_feat], dim=1))
        return temporal_feat * g + temporal_feat


# =========================================================
# 6. DA-TCsiNet
# =========================================================
class DATCsiNet(nn.Module):
    def __init__(
        self,
        in_channels=2,
        height=32,
        width=32,
        compression_ratio=16,
        hidden_dim=256,
        use_tcn=True,
        use_gate=True,
        use_residual=True
    ):
        super().__init__()

        self.use_tcn = use_tcn
        self.use_gate = use_gate
        self.use_residual = use_residual

        self.encoder = CsiEncoder(
            in_channels=in_channels,
            height=height,
            width=width,
            compression_ratio=compression_ratio
        )

        code_dim = self.encoder.code_dim

        self.temporal = MultiScaleTCN(code_dim, hidden_dim)

        self.delta_proj = nn.Sequential(
            nn.Linear(code_dim, hidden_dim),
            nn.ReLU(inplace=True)
        )

        self.gate = DopplerAwareGate(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, code_dim),
            nn.ReLU(inplace=True),
            nn.Linear(code_dim, code_dim)
        )

        self.res_alpha = nn.Parameter(torch.tensor(0.5))

        self.decoder = CsiDecoder(
            out_channels=in_channels,
            height=height,
            width=width,
            code_dim=code_dim
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        z_list = []
        for t in range(T):
            z_list.append(self.encoder(x[:, t]))

        z_seq = torch.stack(z_list, dim=1)
        z_last = z_seq[:, -1]

        if self.use_tcn:
            temporal_feat = self.temporal(z_seq)
        else:

            temporal_feat = self.delta_proj(z_last)

        if T >= 2:
            delta = torch.abs(z_seq[:, -1] - z_seq[:, -2])
        else:
            delta = torch.zeros_like(z_last)

        delta_feat = self.delta_proj(delta)

        if self.use_gate:
            temporal_feat = self.gate(temporal_feat, delta_feat)

        z_temporal = self.proj(temporal_feat)

        if self.use_residual:
            z_final = z_last + self.res_alpha * z_temporal
        else:
            z_final = z_temporal

        return self.decoder(z_final)


# =========================================================
# 7. Metrics / Loss
# =========================================================
def nmse_linear(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    mse = torch.sum(torch.abs(pred_c - target_c) ** 2, dim=1)
    power = torch.sum(torch.abs(target_c) ** 2, dim=1) + 1e-8

    return torch.mean(mse / power)


def corr_complex(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    numerator = torch.abs(torch.sum(torch.conj(target_c) * pred_c, dim=1))

    denominator = torch.sqrt(
        torch.sum(torch.abs(target_c) ** 2, dim=1)
        * torch.sum(torch.abs(pred_c) ** 2, dim=1)
    ) + 1e-8

    return torch.mean(numerator / denominator)


def total_loss_fn(pred, target):
    mse = F.mse_loss(pred, target)
    nmse = nmse_linear(pred, target)
    return mse + 0.1 * nmse


# =========================================================
# 8. Train / Eval
# =========================================================
def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Train", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad()

        pred = model(seq)
        loss = total_loss_fn(pred, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Eval", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        pred = model(seq)
        loss = total_loss_fn(pred, target)

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


# =========================================================
# 9. Complexity
# =========================================================
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def print_complexity(model, model_name="Model"):
    ue_params = count_params(model.encoder)
    total_params = count_params(model)

    print(f"\n===== {model_name} Complexity =====")
    print(f"UE-side Params   : {ue_params:,}")
    print(f"Full Model Params: {total_params:,}")
    print(f"UE / Total       : {ue_params / total_params * 100:.2f}%")


# =========================================================
# 10. Main
# =========================================================
def main():
    print("Device:", DEVICE)

    dataset = CSITemporalDataset(SEQ_PATH, TARGET_PATH)

    sample_seq, sample_target = dataset[0]
    T, C, H, W = sample_seq.shape

    print("T, C, H, W =", T, C, H, W)

    train_len = int(len(dataset) * 0.7)
    val_len = int(len(dataset) * 0.15)
    test_len = len(dataset) - train_len - val_len

    train_set, val_set, test_set = random_split(
        dataset,
        [train_len, val_len, test_len],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = DATCsiNet(
      in_channels=C, height=H, width=W,
      compression_ratio=COMPRESSION_RATIO,
      hidden_dim=HIDDEN_DIM,
      use_tcn=True,
      use_gate=True,
      use_residual=True
    ).to(DEVICE)

    print(model)
    print("Code dim:", model.encoder.code_dim)
    print_complexity(model, "DA-TCsiNet")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_nmse = float("inf")
    best_path = os.path.join(SAVE_DIR, "best_datcsinet.pth")

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_nmse, train_corr = train_one_epoch(
            model, train_loader, optimizer
        )

        val_loss, val_nmse, val_corr = evaluate(
            model, val_loader
        )

        scheduler.step()

        train_nmse_db = 10 * np.log10(train_nmse + 1e-12)
        val_nmse_db = 10 * np.log10(val_nmse + 1e-12)

        print(
            f"Epoch [{epoch:03d}/{EPOCHS}] "
            f"Train Loss: {train_loss:.6f} | "
            f"Train NMSE(dB): {train_nmse_db:.4f} | "
            f"Train Corr: {train_corr:.4f} || "
            f"Val Loss: {val_loss:.6f} | "
            f"Val NMSE(dB): {val_nmse_db:.4f} | "
            f"Val Corr: {val_corr:.4f}"
        )

        if val_nmse < best_val_nmse:
            best_val_nmse = val_nmse

            torch.save({
                "model_state_dict": model.state_dict(),
                "T": T,
                "C": C,
                "H": H,
                "W": W,
                "compression_ratio": COMPRESSION_RATIO,
                "hidden_dim": HIDDEN_DIM,
                "best_val_nmse": best_val_nmse,
            }, best_path)

            print("Saved best model:", best_path)

    print("\nLoading best model for testing...")
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    test_loss, test_nmse, test_corr = evaluate(model, test_loader)

    print("\n========== Final Test Result ==========")
    print(f"Test Loss      : {test_loss:.6f}")
    print(f"Test NMSE      : {test_nmse:.6f}")
    print(f"Test NMSE(dB)  : {10*np.log10(test_nmse + 1e-12):.4f} dB")
    print(f"Test Corr      : {test_corr:.4f}")

    print_complexity(model, "DA-TCsiNet")
    print("========================================")


if __name__ == "__main__":
    main()

Device: cuda
Sequence shape: (423, 4, 2, 32, 32)
Target shape: (423, 2, 32, 32)
T, C, H, W = 4 2 32 32
DATCsiNet(
  (encoder): CsiEncoder(
    (conv): Sequential(
      (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.2, inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): LeakyReLU(negative_slope=0.2, inplace=True)
    )
    (fc): Linear(in_features=32768, out_features=2048, bias=True)
  )
  (temporal): MultiScaleTCN(
    (branch3): Sequential(
      (0): Conv1d(2048, 256, kernel_size=(3,), stride=(1,), padding=(2,))
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (branch5): Sequential(
      (0): Conv1d(2048, 256, ke

Epoch [001/200] Train Loss: 0.164351 | Train NMSE(dB): -1.3716 | Train Corr: 0.6368 || Val Loss: 0.122605 | Val NMSE(dB): -2.6439 | Val Corr: 0.9670
Saved best model: ./datcsinet_ckpt/best_datcsinet.pth


Epoch [002/200] Train Loss: 0.074667 | Train NMSE(dB): -4.7980 | Train Corr: 0.9763 || Val Loss: 0.027001 | Val NMSE(dB): -9.2152 | Val Corr: 0.9887
Saved best model: ./datcsinet_ckpt/best_datcsinet.pth


Epoch [003/200] Train Loss: 0.038474 | Train NMSE(dB): -7.6778 | Train Corr: 0.9919 || Val Loss: 0.018894 | Val NMSE(dB): -10.7658 | Val Corr: 0.9943
Saved best model: ./datcsinet_ckpt/best_datcsinet.pth


Epoch [004/200] Train Loss: 0.021576 | Train NMSE(dB): -10.1896 | Train Corr: 0.9956 || Val Loss: 0.016396 | Val NMSE(dB): -11.3817 | Val Corr: 0.9964
Saved best model: ./datcsinet_ckpt/best_datcsinet.pth


...

Epoch [197/200] Train Loss: 0.000297 | Train NMSE(dB): -28.8029 | Train Corr: 0.9995 || Val Loss: 0.000391 | Val NMSE(dB): -27.6049 | Val Corr: 0.9993
Saved best model: ./datcsinet_ckpt/best_datcsinet.pth


Epoch [198/200] Train Loss: 0.000293 | Train NMSE(dB): -28.8648 | Train Corr: 0.9995 || Val Loss: 0.000390 | Val NMSE(dB): -27.6250 | Val Corr: 0.9993
Saved best model: ./datcsinet_ckpt/best_datcsinet.pth


Epoch [199/200] Train Loss: 0.000295 | Train NMSE(dB): -28.8377 | Train Corr: 0.9995 || Val Loss: 0.000392 | Val NMSE(dB): -27.6000 | Val Corr: 0.9993


Epoch [200/200] Train Loss: 0.000296 | Train NMSE(dB): -28.8140 | Train Corr: 0.9995 || Val Loss: 0.000393 | Val NMSE(dB): -27.5915 | Val Corr: 0.9993

Loading best model for testing...



========== Final Test Result ==========
Test Loss      : 0.000430
Test NMSE      : 0.001904
Test NMSE(dB)  : -27.2023 dB
Test Corr      : 0.9992

===== DA-TCsiNet Complexity =====
UE-side Params   : 67,120,896
Full Model Params: 145,687,987
UE / Total       : 46.07%


# Ablation Studies -1 without Doppler-aware Delta Gate

In [15]:
# =========================================================
# DA-TCsiNet
# Multi-scale TCN + Doppler-aware Delta Gate + Last-frame Residual Fusion
# =========================================================

import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split


# =========================================================
# 1. Config
# =========================================================
SEQ_PATH = "/content/drive/MyDrive/D6_temporal/D6_seq.npy"
TARGET_PATH = "/content/drive/MyDrive/D6_temporal/D6_target.npy"

SAVE_DIR = "./datcsinet_ckpt_woDoppler"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-5

COMPRESSION_RATIO = 16
HIDDEN_DIM = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# 2. Seed
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# =========================================================
# 3. Dataset
# =========================================================
class CSITemporalDataset(Dataset):
    def __init__(self, seq_path, target_path):
        self.seq = np.load(seq_path).astype(np.float32)
        self.target = np.load(target_path).astype(np.float32)

        assert self.seq.ndim == 5, self.seq.shape
        assert self.target.ndim == 4, self.target.shape
        assert len(self.seq) == len(self.target)

        print("Sequence shape:", self.seq.shape)
        print("Target shape:", self.target.shape)

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return torch.from_numpy(self.seq[idx]), torch.from_numpy(self.target[idx])


# =========================================================
# 4. Encoder / Decoder
# =========================================================
class CsiEncoder(nn.Module):
    def __init__(self, in_channels, height, width, compression_ratio=16):
        super().__init__()

        self.height = height
        self.width = width

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
        )

        feature_dim = 32 * height * width
        self.code_dim = max(16, feature_dim // compression_ratio)

        self.fc = nn.Linear(feature_dim, self.code_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(1)
        return self.fc(x)


class CsiDecoder(nn.Module):
    def __init__(self, out_channels, height, width, code_dim):
        super().__init__()

        self.height = height
        self.width = width

        self.fc = nn.Linear(code_dim, 32 * height * width)

        self.decoder = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(16, out_channels, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), 32, self.height, self.width)
        return self.decoder(x)


# =========================================================
# 5. Proposed Temporal Modules
# =========================================================
class MultiScaleTCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()

        self.branch3 = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.branch5 = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=5, padding=4),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.branch_dilated = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=4, dilation=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.fuse = nn.Sequential(
            nn.Conv1d(hidden_dim * 3, hidden_dim, kernel_size=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # x: [B, T, code_dim]
        x = x.transpose(1, 2)  # [B, code_dim, T]

        y1 = self.branch3(x)
        y2 = self.branch5(x)
        y3 = self.branch_dilated(x)

        min_len = min(y1.size(-1), y2.size(-1), y3.size(-1))
        y1 = y1[:, :, -min_len:]
        y2 = y2[:, :, -min_len:]
        y3 = y3[:, :, -min_len:]

        y = torch.cat([y1, y2, y3], dim=1)
        y = self.fuse(y)

        return y[:, :, -1]  # [B, hidden_dim]


class DopplerAwareGate(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )

    def forward(self, temporal_feat, delta_feat):
        g = self.gate(torch.cat([temporal_feat, delta_feat], dim=1))
        return temporal_feat * g + temporal_feat


# =========================================================
# 6. DA-TCsiNet
# =========================================================
class DATCsiNet(nn.Module):
    def __init__(
        self,
        in_channels=2,
        height=32,
        width=32,
        compression_ratio=16,
        hidden_dim=256,
        use_tcn=True,
        use_gate=False,
        use_residual=True
    ):
        super().__init__()

        self.use_tcn = use_tcn
        self.use_gate = use_gate
        self.use_residual = use_residual

        self.encoder = CsiEncoder(
            in_channels=in_channels,
            height=height,
            width=width,
            compression_ratio=compression_ratio
        )

        code_dim = self.encoder.code_dim

        self.temporal = MultiScaleTCN(code_dim, hidden_dim)

        self.delta_proj = nn.Sequential(
            nn.Linear(code_dim, hidden_dim),
            nn.ReLU(inplace=True)
        )

        self.gate = DopplerAwareGate(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, code_dim),
            nn.ReLU(inplace=True),
            nn.Linear(code_dim, code_dim)
        )

        self.res_alpha = nn.Parameter(torch.tensor(0.5))

        self.decoder = CsiDecoder(
            out_channels=in_channels,
            height=height,
            width=width,
            code_dim=code_dim
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        z_list = []
        for t in range(T):
            z_list.append(self.encoder(x[:, t]))

        z_seq = torch.stack(z_list, dim=1)
        z_last = z_seq[:, -1]

        if self.use_tcn:
            temporal_feat = self.temporal(z_seq)
        else:

            temporal_feat = self.delta_proj(z_last)

        if T >= 2:
            delta = torch.abs(z_seq[:, -1] - z_seq[:, -2])
        else:
            delta = torch.zeros_like(z_last)

        delta_feat = self.delta_proj(delta)

        if self.use_gate:
            temporal_feat = self.gate(temporal_feat, delta_feat)

        z_temporal = self.proj(temporal_feat)

        if self.use_residual:
            z_final = z_last + self.res_alpha * z_temporal
        else:
            z_final = z_temporal

        return self.decoder(z_final)


# =========================================================
# 7. Metrics / Loss
# =========================================================
def nmse_linear(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    mse = torch.sum(torch.abs(pred_c - target_c) ** 2, dim=1)
    power = torch.sum(torch.abs(target_c) ** 2, dim=1) + 1e-8

    return torch.mean(mse / power)


def corr_complex(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    numerator = torch.abs(torch.sum(torch.conj(target_c) * pred_c, dim=1))

    denominator = torch.sqrt(
        torch.sum(torch.abs(target_c) ** 2, dim=1)
        * torch.sum(torch.abs(pred_c) ** 2, dim=1)
    ) + 1e-8

    return torch.mean(numerator / denominator)


def total_loss_fn(pred, target):
    mse = F.mse_loss(pred, target)
    nmse = nmse_linear(pred, target)
    return mse + 0.1 * nmse


# =========================================================
# 8. Train / Eval
# =========================================================
def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Train", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad()

        pred = model(seq)
        loss = total_loss_fn(pred, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Eval", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        pred = model(seq)
        loss = total_loss_fn(pred, target)

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


# =========================================================
# 9. Complexity
# =========================================================
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def print_complexity(model, model_name="Model"):
    ue_params = count_params(model.encoder)
    total_params = count_params(model)

    print(f"\n===== {model_name} Complexity =====")
    print(f"UE-side Params   : {ue_params:,}")
    print(f"Full Model Params: {total_params:,}")
    print(f"UE / Total       : {ue_params / total_params * 100:.2f}%")


# =========================================================
# 10. Main
# =========================================================
def main():
    print("Device:", DEVICE)

    dataset = CSITemporalDataset(SEQ_PATH, TARGET_PATH)

    sample_seq, sample_target = dataset[0]
    T, C, H, W = sample_seq.shape

    print("T, C, H, W =", T, C, H, W)

    train_len = int(len(dataset) * 0.7)
    val_len = int(len(dataset) * 0.15)
    test_len = len(dataset) - train_len - val_len

    train_set, val_set, test_set = random_split(
        dataset,
        [train_len, val_len, test_len],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = DATCsiNet(
      in_channels=C, height=H, width=W,
      compression_ratio=COMPRESSION_RATIO,
      hidden_dim=HIDDEN_DIM,
      use_tcn=True,
      use_gate=False,
      use_residual=True
    ).to(DEVICE)

    print(model)
    print("Code dim:", model.encoder.code_dim)
    print_complexity(model, "DA-TCsiNet_woDoppler")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_nmse = float("inf")
    best_path = os.path.join(SAVE_DIR, "best_datcsinet_woDoppler.pth")

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_nmse, train_corr = train_one_epoch(
            model, train_loader, optimizer
        )

        val_loss, val_nmse, val_corr = evaluate(
            model, val_loader
        )

        scheduler.step()

        train_nmse_db = 10 * np.log10(train_nmse + 1e-12)
        val_nmse_db = 10 * np.log10(val_nmse + 1e-12)

        print(
            f"Epoch [{epoch:03d}/{EPOCHS}] "
            f"Train Loss: {train_loss:.6f} | "
            f"Train NMSE(dB): {train_nmse_db:.4f} | "
            f"Train Corr: {train_corr:.4f} || "
            f"Val Loss: {val_loss:.6f} | "
            f"Val NMSE(dB): {val_nmse_db:.4f} | "
            f"Val Corr: {val_corr:.4f}"
        )

        if val_nmse < best_val_nmse:
            best_val_nmse = val_nmse

            torch.save({
                "model_state_dict": model.state_dict(),
                "T": T,
                "C": C,
                "H": H,
                "W": W,
                "compression_ratio": COMPRESSION_RATIO,
                "hidden_dim": HIDDEN_DIM,
                "best_val_nmse": best_val_nmse,
            }, best_path)

            print("Saved best model:", best_path)

    print("\nLoading best model for testing...")
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    test_loss, test_nmse, test_corr = evaluate(model, test_loader)

    print("\n========== Final Test Result ==========")
    print(f"Test Loss      : {test_loss:.6f}")
    print(f"Test NMSE      : {test_nmse:.6f}")
    print(f"Test NMSE(dB)  : {10*np.log10(test_nmse + 1e-12):.4f} dB")
    print(f"Test Corr      : {test_corr:.4f}")

    print_complexity(model, "DA-TCsiNet_woDoppler")
    print("========================================")


if __name__ == "__main__":
    main()

Device: cuda
Sequence shape: (423, 4, 2, 32, 32)
Target shape: (423, 2, 32, 32)
T, C, H, W = 4 2 32 32
DATCsiNet(
  (encoder): CsiEncoder(
    (conv): Sequential(
      (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.2, inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): LeakyReLU(negative_slope=0.2, inplace=True)
    )
    (fc): Linear(in_features=32768, out_features=2048, bias=True)
  )
  (temporal): MultiScaleTCN(
    (branch3): Sequential(
      (0): Conv1d(2048, 256, kernel_size=(3,), stride=(1,), padding=(2,))
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (branch5): Sequential(
      (0): Conv1d(2048, 256, ke

Epoch [001/200] Train Loss: 0.164007 | Train NMSE(dB): -1.3807 | Train Corr: 0.6355 || Val Loss: 0.119144 | Val NMSE(dB): -2.7682 | Val Corr: 0.9680
Saved best model: ./datcsinet_ckpt_woDoppler/best_datcsinet_woDoppler.pth


Epoch [002/200] Train Loss: 0.074390 | Train NMSE(dB): -4.8141 | Train Corr: 0.9777 || Val Loss: 0.025117 | Val NMSE(dB): -9.5294 | Val Corr: 0.9894
Saved best model: ./datcsinet_ckpt_woDoppler/best_datcsinet_woDoppler.pth


Epoch [003/200] Train Loss: 0.038196 | Train NMSE(dB): -7.7092 | Train Corr: 0.9926 || Val Loss: 0.017200 | Val NMSE(dB): -11.1737 | Val Corr: 0.9947
Saved best model: ./datcsinet_ckpt_woDoppler/best_datcsinet_woDoppler.pth


Epoch [004/200] Train Loss: 0.021444 | Train NMSE(dB): -10.2161 | Train Corr: 0.9958 || Val Loss: 0.015524 | Val NMSE(dB): -11.6191 | Val Corr: 0.9966
Saved best model: ./datcsinet_ckpt_woDoppler/best_datcsinet_woDoppler.pth


...

Epoch [197/200] Train Loss: 0.000309 | Train NMSE(dB): -28.6319 | Train Corr: 0.9995 || Val Loss: 0.000407 | Val NMSE(dB): -27.4371 | Val Corr: 0.9993
Saved best model: ./datcsinet_ckpt_woDoppler/best_datcsinet_woDoppler.pth


Epoch [198/200] Train Loss: 0.000305 | Train NMSE(dB): -28.6901 | Train Corr: 0.9995 || Val Loss: 0.000405 | Val NMSE(dB): -27.4584 | Val Corr: 0.9993
Saved best model: ./datcsinet_ckpt_woDoppler/best_datcsinet_woDoppler.pth


Epoch [199/200] Train Loss: 0.000307 | Train NMSE(dB): -28.6640 | Train Corr: 0.9995 || Val Loss: 0.000407 | Val NMSE(dB): -27.4352 | Val Corr: 0.9993


Epoch [200/200] Train Loss: 0.000308 | Train NMSE(dB): -28.6451 | Train Corr: 0.9995 || Val Loss: 0.000408 | Val NMSE(dB): -27.4274 | Val Corr: 0.9993

Loading best model for testing...



========== Final Test Result ==========
Test Loss      : 0.000439
Test NMSE      : 0.001947
Test NMSE(dB)  : -27.1054 dB
Test Corr      : 0.9992

===== DA-TCsiNet_woDoppler Complexity =====
UE-side Params   : 67,120,896
Full Model Params: 145,687,987
UE / Total       : 46.07%


# Ablation Studies -2 without Last-frame Residual Fusion

In [16]:
# =========================================================
# DA-TCsiNet
# Multi-scale TCN + Doppler-aware Delta Gate + Last-frame Residual Fusion
# =========================================================

import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split


# =========================================================
# 1. Config
# =========================================================
SEQ_PATH = "/content/drive/MyDrive/D6_temporal/D6_seq.npy"
TARGET_PATH = "/content/drive/MyDrive/D6_temporal/D6_target.npy"

SAVE_DIR = "./datcsinet_2_ckpt_woResidualFusion"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-5

COMPRESSION_RATIO = 16
HIDDEN_DIM = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# 2. Seed
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# =========================================================
# 3. Dataset
# =========================================================
class CSITemporalDataset(Dataset):
    def __init__(self, seq_path, target_path):
        self.seq = np.load(seq_path).astype(np.float32)
        self.target = np.load(target_path).astype(np.float32)

        assert self.seq.ndim == 5, self.seq.shape
        assert self.target.ndim == 4, self.target.shape
        assert len(self.seq) == len(self.target)

        print("Sequence shape:", self.seq.shape)
        print("Target shape:", self.target.shape)

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return torch.from_numpy(self.seq[idx]), torch.from_numpy(self.target[idx])


# =========================================================
# 4. Encoder / Decoder
# =========================================================
class CsiEncoder(nn.Module):
    def __init__(self, in_channels, height, width, compression_ratio=16):
        super().__init__()

        self.height = height
        self.width = width

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
        )

        feature_dim = 32 * height * width
        self.code_dim = max(16, feature_dim // compression_ratio)

        self.fc = nn.Linear(feature_dim, self.code_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(1)
        return self.fc(x)


class CsiDecoder(nn.Module):
    def __init__(self, out_channels, height, width, code_dim):
        super().__init__()

        self.height = height
        self.width = width

        self.fc = nn.Linear(code_dim, 32 * height * width)

        self.decoder = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(16, out_channels, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), 32, self.height, self.width)
        return self.decoder(x)


# =========================================================
# 5. Proposed Temporal Modules
# =========================================================
class MultiScaleTCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()

        self.branch3 = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.branch5 = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=5, padding=4),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.branch_dilated = nn.Sequential(
            nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=4, dilation=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.fuse = nn.Sequential(
            nn.Conv1d(hidden_dim * 3, hidden_dim, kernel_size=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # x: [B, T, code_dim]
        x = x.transpose(1, 2)  # [B, code_dim, T]

        y1 = self.branch3(x)
        y2 = self.branch5(x)
        y3 = self.branch_dilated(x)

        min_len = min(y1.size(-1), y2.size(-1), y3.size(-1))
        y1 = y1[:, :, -min_len:]
        y2 = y2[:, :, -min_len:]
        y3 = y3[:, :, -min_len:]

        y = torch.cat([y1, y2, y3], dim=1)
        y = self.fuse(y)

        return y[:, :, -1]  # [B, hidden_dim]


class DopplerAwareGate(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )

    def forward(self, temporal_feat, delta_feat):
        g = self.gate(torch.cat([temporal_feat, delta_feat], dim=1))
        return temporal_feat * g + temporal_feat


# =========================================================
# 6. DA-TCsiNet
# =========================================================
class DATCsiNet(nn.Module):
    def __init__(
        self,
        in_channels=2,
        height=32,
        width=32,
        compression_ratio=16,
        hidden_dim=256,
        use_tcn=True,
        use_gate=True,
        use_residual=False
    ):
        super().__init__()

        self.use_tcn = use_tcn
        self.use_gate = use_gate
        self.use_residual = use_residual

        self.encoder = CsiEncoder(
            in_channels=in_channels,
            height=height,
            width=width,
            compression_ratio=compression_ratio
        )

        code_dim = self.encoder.code_dim

        self.temporal = MultiScaleTCN(code_dim, hidden_dim)

        self.delta_proj = nn.Sequential(
            nn.Linear(code_dim, hidden_dim),
            nn.ReLU(inplace=True)
        )

        self.gate = DopplerAwareGate(hidden_dim)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, code_dim),
            nn.ReLU(inplace=True),
            nn.Linear(code_dim, code_dim)
        )

        self.res_alpha = nn.Parameter(torch.tensor(0.5))

        self.decoder = CsiDecoder(
            out_channels=in_channels,
            height=height,
            width=width,
            code_dim=code_dim
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        z_list = []
        for t in range(T):
            z_list.append(self.encoder(x[:, t]))

        z_seq = torch.stack(z_list, dim=1)
        z_last = z_seq[:, -1]

        if self.use_tcn:
            temporal_feat = self.temporal(z_seq)
        else:

            temporal_feat = self.delta_proj(z_last)

        if T >= 2:
            delta = torch.abs(z_seq[:, -1] - z_seq[:, -2])
        else:
            delta = torch.zeros_like(z_last)

        delta_feat = self.delta_proj(delta)

        if self.use_gate:
            temporal_feat = self.gate(temporal_feat, delta_feat)

        z_temporal = self.proj(temporal_feat)

        if self.use_residual:
            z_final = z_last + self.res_alpha * z_temporal
        else:
            z_final = z_temporal

        return self.decoder(z_final)


# =========================================================
# 7. Metrics / Loss
# =========================================================
def nmse_linear(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    mse = torch.sum(torch.abs(pred_c - target_c) ** 2, dim=1)
    power = torch.sum(torch.abs(target_c) ** 2, dim=1) + 1e-8

    return torch.mean(mse / power)


def corr_complex(pred, target):
    pred_c = (pred[:, 0] - 0.5) + 1j * (pred[:, 1] - 0.5)
    target_c = (target[:, 0] - 0.5) + 1j * (target[:, 1] - 0.5)

    pred_c = pred_c.flatten(1)
    target_c = target_c.flatten(1)

    numerator = torch.abs(torch.sum(torch.conj(target_c) * pred_c, dim=1))

    denominator = torch.sqrt(
        torch.sum(torch.abs(target_c) ** 2, dim=1)
        * torch.sum(torch.abs(pred_c) ** 2, dim=1)
    ) + 1e-8

    return torch.mean(numerator / denominator)


def total_loss_fn(pred, target):
    mse = F.mse_loss(pred, target)
    nmse = nmse_linear(pred, target)
    return mse + 0.1 * nmse


# =========================================================
# 8. Train / Eval
# =========================================================
def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Train", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad()

        pred = model(seq)
        loss = total_loss_fn(pred, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0
    total_nmse = 0
    total_corr = 0

    for seq, target in tqdm(loader, desc="Eval", leave=False):
        seq = seq.to(DEVICE)
        target = target.to(DEVICE)

        pred = model(seq)
        loss = total_loss_fn(pred, target)

        total_loss += loss.item()
        total_nmse += nmse_linear(pred, target).item()
        total_corr += corr_complex(pred, target).item()

    n = len(loader)
    return total_loss / n, total_nmse / n, total_corr / n


# =========================================================
# 9. Complexity
# =========================================================
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def print_complexity(model, model_name="Model"):
    ue_params = count_params(model.encoder)
    total_params = count_params(model)

    print(f"\n===== {model_name} Complexity =====")
    print(f"UE-side Params   : {ue_params:,}")
    print(f"Full Model Params: {total_params:,}")
    print(f"UE / Total       : {ue_params / total_params * 100:.2f}%")


# =========================================================
# 10. Main
# =========================================================
def main():
    print("Device:", DEVICE)

    dataset = CSITemporalDataset(SEQ_PATH, TARGET_PATH)

    sample_seq, sample_target = dataset[0]
    T, C, H, W = sample_seq.shape

    print("T, C, H, W =", T, C, H, W)

    train_len = int(len(dataset) * 0.7)
    val_len = int(len(dataset) * 0.15)
    test_len = len(dataset) - train_len - val_len

    train_set, val_set, test_set = random_split(
        dataset,
        [train_len, val_len, test_len],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = DATCsiNet(
      in_channels=C, height=H, width=W,
      compression_ratio=COMPRESSION_RATIO,
      hidden_dim=HIDDEN_DIM,
      use_tcn=True,
      use_gate=True,
      use_residual=False
    ).to(DEVICE)

    print(model)
    print("Code dim:", model.encoder.code_dim)
    print_complexity(model, "DA-TCsiNet_woResidualFusion")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_nmse = float("inf")
    best_path = os.path.join(SAVE_DIR, "best_datcsinet_woResidualFusion.pth")

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_nmse, train_corr = train_one_epoch(
            model, train_loader, optimizer
        )

        val_loss, val_nmse, val_corr = evaluate(
            model, val_loader
        )

        scheduler.step()

        train_nmse_db = 10 * np.log10(train_nmse + 1e-12)
        val_nmse_db = 10 * np.log10(val_nmse + 1e-12)

        print(
            f"Epoch [{epoch:03d}/{EPOCHS}] "
            f"Train Loss: {train_loss:.6f} | "
            f"Train NMSE(dB): {train_nmse_db:.4f} | "
            f"Train Corr: {train_corr:.4f} || "
            f"Val Loss: {val_loss:.6f} | "
            f"Val NMSE(dB): {val_nmse_db:.4f} | "
            f"Val Corr: {val_corr:.4f}"
        )

        if val_nmse < best_val_nmse:
            best_val_nmse = val_nmse

            torch.save({
                "model_state_dict": model.state_dict(),
                "T": T,
                "C": C,
                "H": H,
                "W": W,
                "compression_ratio": COMPRESSION_RATIO,
                "hidden_dim": HIDDEN_DIM,
                "best_val_nmse": best_val_nmse,
            }, best_path)

            print("Saved best model:", best_path)

    print("\nLoading best model for testing...")
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    test_loss, test_nmse, test_corr = evaluate(model, test_loader)

    print("\n========== Final Test Result ==========")
    print(f"Test Loss      : {test_loss:.6f}")
    print(f"Test NMSE      : {test_nmse:.6f}")
    print(f"Test NMSE(dB)  : {10*np.log10(test_nmse + 1e-12):.4f} dB")
    print(f"Test Corr      : {test_corr:.4f}")

    print_complexity(model, "DA-TCsiNet_woResidualFusion")
    print("========================================")


if __name__ == "__main__":
    main()

Device: cuda
Sequence shape: (423, 4, 2, 32, 32)
Target shape: (423, 2, 32, 32)
T, C, H, W = 4 2 32 32
DATCsiNet(
  (encoder): CsiEncoder(
    (conv): Sequential(
      (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.2, inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): LeakyReLU(negative_slope=0.2, inplace=True)
    )
    (fc): Linear(in_features=32768, out_features=2048, bias=True)
  )
  (temporal): MultiScaleTCN(
    (branch3): Sequential(
      (0): Conv1d(2048, 256, kernel_size=(3,), stride=(1,), padding=(2,))
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (branch5): Sequential(
      (0): Conv1d(2048, 256, ke

Epoch [001/200] Train Loss: 0.157152 | Train NMSE(dB): -1.5661 | Train Corr: 0.6819 || Val Loss: 0.083063 | Val NMSE(dB): -4.3349 | Val Corr: 0.9695
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


Epoch [002/200] Train Loss: 0.071150 | Train NMSE(dB): -5.0075 | Train Corr: 0.9807 || Val Loss: 0.026014 | Val NMSE(dB): -9.3769 | Val Corr: 0.9899
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


Epoch [003/200] Train Loss: 0.037402 | Train NMSE(dB): -7.8004 | Train Corr: 0.9931 || Val Loss: 0.018832 | Val NMSE(dB): -10.7802 | Val Corr: 0.9952
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


Epoch [004/200] Train Loss: 0.021012 | Train NMSE(dB): -10.3046 | Train Corr: 0.9961 || Val Loss: 0.017919 | Val NMSE(dB): -10.9960 | Val Corr: 0.9968
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


...

Epoch [197/200] Train Loss: 0.000727 | Train NMSE(dB): -24.9166 | Train Corr: 0.9985 || Val Loss: 0.000732 | Val NMSE(dB): -24.8892 | Val Corr: 0.9985
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


Epoch [198/200] Train Loss: 0.000710 | Train NMSE(dB): -25.0188 | Train Corr: 0.9985 || Val Loss: 0.000731 | Val NMSE(dB): -24.8893 | Val Corr: 0.9985
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


Epoch [199/200] Train Loss: 0.000721 | Train NMSE(dB): -24.9528 | Train Corr: 0.9985 || Val Loss: 0.000731 | Val NMSE(dB): -24.8894 | Val Corr: 0.9985
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth


Epoch [200/200] Train Loss: 0.000727 | Train NMSE(dB): -24.9178 | Train Corr: 0.9985 || Val Loss: 0.000731 | Val NMSE(dB): -24.8895 | Val Corr: 0.9985
Saved best model: ./datcsinet_2_ckpt_woResidualFusion/best_datcsinet_woResidualFusion.pth

Loading best model for testing...



========== Final Test Result ==========
Test Loss      : 0.000821
Test NMSE      : 0.003637
Test NMSE(dB)  : -24.3922 dB
Test Corr      : 0.9983

===== DA-TCsiNet_woResidualFusion Complexity =====
UE-side Params   : 67,120,896
Full Model Params: 145,687,987
UE / Total       : 46.07%
